# 00b - Effective surface area MAF: the footprint weighted by its weak-lensing source density

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-29
- Kernel: conda_py313_opsim53
- Context: series `10_DESCMAFDEPTHANDGALCOUNT`. Notebook 05 built a seeing-aware weak-lensing effective source density `n_eff` (model of notebook 04, Chang et al. 2013) and integrated it over the footprint. **This notebook packages that computation as a real MAF metric, `EffectiveSurfaceAreaMetric`**, that can be run on any OpSim database with a `HealpixSlicer`, and applies it to the 7 simulations of the series (`shrink_fp_dust_{0.050, 0.080, 0.120, 0.150, 0.199, 0.250}_v5.3.6_10yrs.db` and `baseline_v5.3.6_11yrs.db`, E(B-V) threshold 0.200).
- Old content of this notebook (signatures and sources of the `rubin_sim` metrics): `00_DepthsAndCountsMetrics_old.ipynb`.

## Notebook overview

**Definition.** The effective surface area of a footprint is the geometric area of each pixel, weighted by the source density of the pixel relative to a reference density $n_{\rm ref}$:

$$A_{\rm eff} = \sum_{\rm pixels} \Omega_{\rm pix}\,\frac{n_{\rm eff}(\hat n)}{n_{\rm ref}} = \frac{N_{\rm eff,tot}}{n_{\rm ref}}, \qquad N_{\rm eff,tot}=\int_{\rm footprint} n_{\rm eff}\,d\Omega .$$

**Meaning.** For a shape-noise dominated shear measurement the noise variance scales as $\sigma_e^2/(n_{\rm eff}\,A)$. A survey of area $A$ and density $n_{\rm eff}$ therefore has the same shape noise as a survey of area $A_{\rm eff}$ at the fixed density $n_{\rm ref}$: $A_{\rm eff}$ is one number that combines the size of the footprint and its depth / seeing quality. $A_{\rm eff}<A_{\rm geo}$ when the mean $n_{\rm eff}$ is below $n_{\rm ref}$. The statistical signal-to-noise scales as $\sqrt{A_{\rm eff}}$.

**Choice of the reference density.** `N_REF = 27` arcmin$^{-2}$ (Section 2), the year-10 source density of the DESC SRD; edit it if another reference is wanted. It only rescales $A_{\rm eff}$: the ratio to the baseline does not depend on it.

**The MAF (Section 4).** `EffectiveSurfaceAreaMetric.run()` is evaluated in every HEALPix pixel and returns the contribution of the pixel, $a_{\rm eff}=\Omega_{\rm pix}\,n_{\rm eff}/n_{\rm ref}$ in deg$^2$, so that **the sum of the map is $A_{\rm eff}$** whatever `nside`. In a pixel it:
1. returns the bad value if `E(B-V)` is above the cut (`extinction_cut`, set to the threshold of the run);
2. computes the dust-corrected coadded 5-sigma depth of each band (`ExgalM5WithCuts` with all its cuts disabled, as in notebook 05) and the coadded effective seeing $\sqrt{\langle {\rm FWHM}^2\rangle}$ of each band;
3. returns the bad value if the coadded depth of the shape band (`i`) is below `depth_cut` = 25.9 (footprint criterion of notebook 03);
4. evaluates the `n_eff` model of notebook 05 (`generic`, or `R2cut` with the resolution cut $R_2>0.3$) and returns $\Omega_{\rm pix}\,n_{\rm eff}/n_{\rm ref}$.

**Differences with notebook 05.** The footprint is defined inside the MAF (E(B-V) cut of the run and `i`-band depth cut) instead of being read from the cache of notebook 03; the minimum exposure-time cut of notebook 03 is not applied. Section 7 checks that the two computations agree on the common pixels.

**Outputs**: MAF results and cached maps in `data_00_EFFSURFACE/`, figures in `figs_00_EFFSURFACE/`. Maps are plotted as the dimensionless weight $w=a_{\rm eff}/\Omega_{\rm pix}=n_{\rm eff}/n_{\rm ref}$, which does not depend on `nside`.

## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics import BaseMetric
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import ExgalM5WithCuts

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "EFFSURFACE"
data_dir = f"data_00b_{NB_TAG}"
figs_dir = f"figs_00b_{NB_TAG}"
DIR05 = "data_05_NEFFMAPS"  # caches of notebook 05 (used for the validation of Section 7 only)
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

OPSIM_DIR = "/Users/dagoret/DATA/OpSim"
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
EXTINCTION_CUT = dict(
    RUNS_INFO
)  # E(B-V) cut of the metric = threshold that defines the footprint of each run
REF_RUN = "baseline_v5.3.6_10yrs"
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


NSIDE = 64
MAX_NIGHT = 10 * 365.25 + 0.5  # first 10 years of every run
BANDS = ("r", "i")  # bands of the joint shape measurement
SHAPE_BAND = "i"  # band on which the depth cut and the resolution cut are applied
SEEING_COL = "seeing" + "FwhmGeom"  # geometric FWHM, as in notebook 05
DEPTH_CUT = (
    25.9  # minimum dust-corrected coadded depth of the shape band (footprint criterion of notebook 03)
)
N_REF = 27.0  # reference source density [arcmin^-2] (DESC SRD year 10); only rescales A_eff
FORCE_RECOMPUTE = False  # True -> redo the MAF runs

# --- n_eff model (Chang et al. 2013), identical to notebooks 04 and 05
SIGMA_SN = 0.26
ABC = (1.58, 5.03, 0.39)
BLEND = {1.0: (0.50, 1.9e-3), 2.0: (0.68, 5.5e-3), 3.0: (0.62, 1.4e-2)}
BLEND_D = 2.0
POP = dict(rh0=0.35, xi=0.13, sig_lnr=0.45, size_scale=1.0)  # ASSUMED galaxy population, as in notebook 04
OFFSET = dict(r=0.4, i=0.0, z=-0.1)  # ASSUMED colours m_band = m_i + OFFSET[band]
R2_MIN = 0.3  # HSC-like resolution cut R2 = R / (1 + R) > R2_MIN
VARIANTS = {"generic": None, "R2cut": R2_MIN / (1.0 - R2_MIN)}  # variant -> minimum R on the shape band
POP_TAG = "rh%.2f_xi%.2f_sl%.2f_ss%.2f" % (POP["rh0"], POP["xi"], POP["sig_lnr"], POP["size_scale"])
BANDS_TAG = "".join(BANDS)

pix_area = hp.nside2pixarea(NSIDE, degrees=True)  # deg^2
pix_arcmin2 = pix_area * 3600.0


def sql_bands():
    bands = " or ".join(f"band = {b!r}" for b in BANDS)
    return f"scheduler_note not like 'DD%' and night <= {MAX_NIGHT!r} and ({bands})"


print(
    f"nside={NSIDE}, pixel area = {pix_area:.4f} deg^2 = {pix_arcmin2:.1f} arcmin^2; n_ref = {N_REF} arcmin^-2"
)
print("sql:", sql_bands())

## 3. The n_eff model, vectorized over the pixels

Same code as Section 3 of notebook 05 (formulas of Chang et al. 2013, joint `r + i` fit, selection `sigma_m < sigma_SN`, optional resolution cut on the shape band, blending). It is called by the MAF with one-element arrays; it is kept in this notebook so that the metric is self-contained.

In [ ]:
def dndm_i(m):
    # differential counts [arcmin^-2 mag^-1] from N(<i) = 46 * 10**(0.31 * (i - 25))  (LSST Science Book)
    return 46.0 * 0.31 * np.log(10.0) * 10 ** (0.31 * (m - 25.0))


def sigma_m(nu, R, abc=ABC):
    a, b, c = abc
    return a / nu * (1.0 + (b / R) ** c)


def neff_map(
    m5,
    fwhm,
    bands=BANDS,
    shape_band=SHAPE_BAND,
    k=1.0,
    R_min=None,
    blend_d=BLEND_D,
    pop=POP,
    dm=0.05,
    nz=41,
    chunk=200,
):
    # m5, fwhm: dict band -> 1-D array over the pixels (NaN = no visit). Returns n_raw and n_eff [arcmin^-2] (NaN where undefined).
    m_i = np.arange(20.0, 28.5, dm) + dm / 2
    z = np.linspace(-3.5, 3.5, nz)
    wz = np.exp(-0.5 * z**2)
    wz /= wz.sum()
    rh = pop["size_scale"] * pop["rh0"] * 10 ** (-pop["xi"] * (m_i - 24.0))
    r_gal = (
        1.46 * rh[:, None] * np.exp(pop["sig_lnr"] * z[None, :])
    )  # (nmag, nz), second-moment radius of a disk
    dN = (dndm_i(m_i) * dm)[:, None] * wz[None, :]
    npix = len(m5[shape_band])
    ok = np.isfinite(m5[shape_band]) & np.isfinite(fwhm[shape_band]) & (fwhm[shape_band] > 0)
    m5v, fwv = {}, {}
    for b in bands:  # missing band -> hopelessly shallow, zero weight in the joint fit
        good = np.isfinite(m5[b]) & np.isfinite(fwhm[b]) & (fwhm[b] > 0)
        m5v[b] = np.where(good, m5[b], 0.0)
        fwv[b] = np.where(good, fwhm[b], 1.0)
    n_raw = np.full(npix, np.nan)
    n_eff = np.full(npix, np.nan)
    idx = np.flatnonzero(ok)
    for s in range(0, len(idx), chunk):
        ii = idx[s : s + chunk]
        inv = 0.0
        for b in bands:
            sig = (fwv[b][ii] / 2.3548)[:, None, None]
            r_psf = np.sqrt(2.0) * sig
            r_ap = 1.64 * np.sqrt(r_gal[None] ** 2 + r_psf**2)
            nu_ps = 5.0 * 10 ** (0.4 * (m5v[b][ii][:, None, None] - (m_i + OFFSET[b])[None, :, None]))
            nu = 0.9 * nu_ps * 2.0 * sig / r_ap
            R = r_gal[None] ** 2 / r_psf**2
            inv = inv + 1.0 / sigma_m(nu, R) ** 2
            if b == shape_band:
                R_shape = R
        sm = 1.0 / np.sqrt(inv)
        sel = sm < k * SIGMA_SN
        if R_min is not None:
            sel = sel & (R_shape >= R_min)
        raw = (dN[None] * sel).sum(axis=(1, 2))
        n0 = (dN[None] * sel * SIGMA_SN**2 / (SIGMA_SN**2 + sm**2)).sum(axis=(1, 2))
        fb = 0.0
        if blend_d is not None:
            eta, mu = BLEND[blend_d]
            fb = eta * np.log(1.0 + mu * raw)
        n_raw[ii] = raw
        n_eff[ii] = n0 * (1.0 - fb)
    return n_raw, n_eff

**Validation against notebook 04**: an LSST-like pixel (10-year coadd depths `r = 27.5`, `i = 26.8`, seeing 0.8 arcsec in both bands) must give `n_eff = 36.1` (generic) and `31.9` (with the resolution cut applied on the `r` band, the band used for the cut in notebook 04).

In [ ]:
one = lambda v: np.array([v])
m5_test = dict(r=one(27.5), i=one(26.8))
fw_test = dict(r=one(0.8), i=one(0.8))
gen = neff_map(m5_test, fw_test)[1][0]
cut = neff_map(m5_test, fw_test, shape_band="r", R_min=R2_MIN / (1.0 - R2_MIN))[1][0]
print(f"generic: {gen:.2f} (notebook 04: 36.1)   R2 > 0.3 on r: {cut:.2f} (notebook 04: 31.9)")

## 4. The MAF metric `EffectiveSurfaceAreaMetric`

The metric asks MAF for the columns `fiveSigmaDepth`, `band` and the seeing column, and for the `DustMap` (which provides `slice_point['ebv']`). Inside `run()`, the coadded dust-corrected depth of each band is delegated to `ExgalM5WithCuts` with all its cuts disabled, exactly as in Section 4 of notebook 05. The value returned is the contribution of the pixel to the effective area, in deg$^2$; the bad value (-666) marks the pixels outside the footprint.

In [ ]:
class EffectiveSurfaceAreaMetric(BaseMetric):
    # Contribution of a HEALPix pixel to the effective surface area [deg^2]:
    #     a_eff = Omega_pix * n_eff / n_ref
    # The sum of the map over the pixels is A_eff = N_eff_tot / n_ref.
    # n_eff comes from the seeing- and depth-aware weak-lensing model of notebooks 04 and 05.
    def __init__(
        self,
        nside=NSIDE,
        n_ref=N_REF,
        bands=BANDS,
        shape_band=SHAPE_BAND,
        r_min=None,
        extinction_cut=0.2,
        depth_cut=DEPTH_CUT,
        m5_col="fiveSigmaDepth",
        filter_col="band",
        seeing_col=SEEING_COL,
        metric_name="EffectiveSurfaceArea",
        **kwargs,
    ):
        self.nside = nside
        self.pix_area = hp.nside2pixarea(nside, degrees=True)  # deg^2
        self.n_ref = float(n_ref)
        self.bands = tuple(bands)
        self.shape_band = shape_band
        self.r_min = r_min  # None -> generic n_eff; a value -> minimum resolution R on the shape band
        self.extinction_cut = extinction_cut
        self.depth_cut = depth_cut
        self.filter_col = filter_col
        self.seeing_col = seeing_col
        # dust-corrected coadded depth of one band, all the ExgalM5WithCuts cuts disabled
        self.exgal = {
            b: ExgalM5WithCuts(
                lsst_filter=b, extinction_cut=99.0, n_filters=1, depth_cut=0.0, metric_name="ExgalM5_" + b
            )
            for b in self.bands
        }
        super().__init__(
            col=[m5_col, filter_col, seeing_col],
            metric_name=metric_name,
            maps=["DustMap"],
            units="deg2",
            **kwargs,
        )

    def run(self, data_slice, slice_point=None):
        # 1. dust cut
        if slice_point["ebv"] > self.extinction_cut:
            return self.badval
        # 2. coadded depth and effective seeing, band by band
        m5, fwhm = {}, {}
        for b in self.bands:
            d = data_slice[data_slice[self.filter_col] == b]
            if len(d) == 0:
                m5[b], fwhm[b] = np.nan, np.nan
                continue
            m5b = float(self.exgal[b].run(d, slice_point))
            m5[b] = m5b if m5b > -600.0 else np.nan
            fwhm[b] = float(np.sqrt(np.mean(np.asarray(d[self.seeing_col], dtype=float) ** 2)))
        # 3. depth cut on the shape band
        if not np.isfinite(m5[self.shape_band]) or m5[self.shape_band] < self.depth_cut:
            return self.badval
        # 4. n_eff model, then weight relative to the reference density
        _, n_eff = neff_map(
            {b: np.array([m5[b]]) for b in self.bands},
            {b: np.array([fwhm[b]]) for b in self.bands},
            bands=self.bands,
            shape_band=self.shape_band,
            R_min=self.r_min,
        )
        if not np.isfinite(n_eff[0]):
            return self.badval
        return self.pix_area * float(n_eff[0]) / self.n_ref

**Smoke test on a synthetic pixel.** 200 visits per band with per-visit depths chosen so that the coadd is `r = 27.5`, `i = 26.8` (coadd = per-visit depth + 1.25 log10(N)), seeing 0.8 arcsec, no dust. The metric must return `pix_area * 36.1 / N_REF`, the `n_eff` value of notebook 04.

In [ ]:
n_vis = 200
fake = np.zeros(2 * n_vis, dtype=[("fiveSigmaDepth", "f8"), ("band", "U1"), (SEEING_COL, "f8")])
for k, (b, m5_target) in enumerate((("r", 27.5), ("i", 26.8))):
    sl = slice(k * n_vis, (k + 1) * n_vis)
    fake["fiveSigmaDepth"][sl] = m5_target - 1.25 * np.log10(n_vis)
    fake["band"][sl] = b
    fake[SEEING_COL][sl] = 0.8

metric_test = EffectiveSurfaceAreaMetric()
a_test = metric_test.run(fake, {"ebv": 0.0})
print(f"a_eff = {a_test:.5f} deg2  ->  n_eff = {a_test * N_REF / pix_area:.2f} arcmin^-2 (notebook 04: 36.1)")
print("above the dust cut ->", metric_test.run(fake, {"ebv": 0.5}), "(bad value)")

## 5. Run the MAF on the 7 simulations

For each run, one `MetricBundle` per n_eff variant (`generic`, `R2cut`), with the E(B-V) cut of the run, on the first 10 years, non-DDF visits, bands `r` and `i`. The maps are cached (the file name contains the seeing column, the bands, the population, the depth cut and `N_REF`). Expect about a minute per run and variant for the first execution: the model is evaluated pixel by pixel.

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def bundle_to_masked(bundle):
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def save_map(path, m):
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m))


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def cache_path(variant, run_name):
    tag = run_name.replace(".", "_")
    return join(
        data_dir,
        f"EFFSURF_{variant}_{tag}_{SEEING_COL}_{BANDS_TAG}_{POP_TAG}_blend{BLEND_D:.0f}_m5cut{DEPTH_CUT:g}_nref{N_REF:g}_nside{NSIDE}.npz",
    )


def load_or_run(run_name):
    paths = {v: cache_path(v, run_name) for v in VARIANTS}
    todo = [v for v in VARIANTS if FORCE_RECOMPUTE or not isfile(paths[v])]
    out = {v: load_map(paths[v]) for v in VARIANTS if v not in todo}
    if not todo:
        print(f"[{run_name}] loaded cached maps")
        return out
    dbpath = get_db_path(run_name)
    print(f"[{run_name}] computing {todo} on {dbpath}")
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundles = {}
    for v in todo:
        metric = EffectiveSurfaceAreaMetric(
            extinction_cut=EXTINCTION_CUT[run_name],
            r_min=VARIANTS[v],
            metric_name="EffectiveSurfaceArea_" + v,
        )
        bundles[v] = mb.MetricBundle(
            metric,
            slicer,
            sql_bands(),
            maps_list=[dustmap],
            run_name=run_name,
            info_label=f"EffectiveSurfaceArea {v} 10yr nonDD",
        )
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    for v, bundle in bundles.items():
        out[v] = bundle_to_masked(bundle)
        save_map(paths[v], out[v])
    return out


_by_run = {run_name: load_or_run(run_name) for run_name in RUN_NAMES}
eff_maps = {
    v: {run_name: _by_run[run_name][v] for run_name in RUN_NAMES} for v in VARIANTS
}  # a_eff [deg2] per pixel
weight_maps = {v: {r: m / pix_area for r, m in eff_maps[v].items()} for v in VARIANTS}  # w = n_eff / n_ref

## 6. Effective surface area per run

`A_geo` is the geometric area of the footprint (number of valid pixels times the pixel area), `A_eff` the sum of the map. `A_eff / A_geo = <n_eff> / n_ref`. `N_eff` is the total effective number of galaxies, `A_eff * n_ref`. `sqrt(A_eff) / baseline` is the shape-noise signal-to-noise proxy.

In [ ]:
def per_run_summary(maps):
    rows = []
    for run_name, ebv in RUNS_INFO:
        a = maps[run_name].compressed()
        rows.append(
            {
                "run": run_name,
                "E(B-V) cut": ebv,
                "n_pixels": a.size,
                "area_deg2": a.size * pix_area,
                "A_eff [deg2]": a.sum(),
                "A_eff / A_geo": a.sum() / (a.size * pix_area) if a.size else np.nan,
                "N_eff [1e6]": a.sum() * N_REF * 3600.0 / 1e6,
            }
        )
    df = pd.DataFrame(rows).set_index("run")
    df["A_eff / baseline"] = df["A_eff [deg2]"] / df.loc[REF_RUN, "A_eff [deg2]"]
    df["A_geo / baseline"] = df["area_deg2"] / df.loc[REF_RUN, "area_deg2"]
    df["sqrt(A_eff) / baseline"] = np.sqrt(df["A_eff / baseline"])
    return df


summaries = {}
for variant in VARIANTS:
    summaries[variant] = per_run_summary(eff_maps[variant])
    summaries[variant].to_csv(join(data_dir, f"EffectiveSurfaceArea_{variant}_per_run_summary.csv"))
    print(f"Effective surface area, variant {variant} (N_REF = {N_REF:g} arcmin^-2):")
    display(summaries[variant].round(4))

## 7. Validation against notebook 05

Where a pixel is valid both here and in the `n_eff` cache of notebook 05 (same model, footprint from notebook 03), the density recovered from the MAF, `a_eff * N_REF / pix_area`, must be equal to the cached `n_eff`. Pixels present in only one of the two maps come from the footprint definitions (the minimum exposure-time cut of notebook 03 is not applied in the MAF). Rows are left empty when the cache of notebook 05 is missing.

In [ ]:
def neff_cache_path_05(variant, run_name):
    tag = run_name.replace(".", "_")
    return join(
        DIR05,
        f"NEFF_{variant}_{tag}_{SEEING_COL}_{BANDS_TAG}_{POP_TAG}_blend{BLEND_D:.0f}_nside{NSIDE}.npz",
    )


rows = []
for variant in VARIANTS:
    for run_name in RUN_NAMES:
        row = {"variant": variant, "run": run_name}
        path = neff_cache_path_05(variant, run_name)
        if isfile(path):
            ref = load_map(path)  # n_eff [arcmin^-2] of notebook 05
            mine = eff_maps[variant][run_name]
            vr, vm = ~np.ma.getmaskarray(ref), ~np.ma.getmaskarray(mine)
            both = vr & vm
            n_here = np.ma.getdata(mine)[both] * N_REF / pix_area
            n_05 = np.ma.getdata(ref)[both]
            rel = n_here / n_05 - 1.0
            row.update(
                {
                    "pixels nb05": int(vr.sum()),
                    "pixels here": int(vm.sum()),
                    "common": int(both.sum()),
                    "only nb05": int((vr & ~vm).sum()),
                    "only here": int((vm & ~vr).sum()),
                    "max |n_eff here / n_eff nb05 - 1|": np.abs(rel).max() if rel.size else np.nan,
                }
            )
        rows.append(row)
validation = pd.DataFrame(rows).set_index(["variant", "run"])
validation.to_csv(join(data_dir, "validation_vs_notebook05.csv"))
display(validation)

## 8. Maps and histograms

The maps show the weight $w=a_{\rm eff}/\Omega_{\rm pix}=n_{\rm eff}/n_{\rm ref}$ (dimensionless, independent of `nside`); the sum of $a_{\rm eff}$ over the pixels is $A_{\rm eff}$. Histograms are weighted by the pixel area.

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {EXTINCTION_CUT[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def as_healpy(m):
    return np.ma.filled(m, hp.UNSEEN)


def plot_all_maps(maps, tag, title, unit="w = n_eff / n_ref", cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked, nothing to plot")
        return
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, ebv) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, title, xlabel="w = n_eff / n_ref", nbins=80):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, ebv), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg²]")
    axs[1].set_ylabel("Area with value ≥ x [deg²]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


for variant in VARIANTS:
    plot_all_maps(weight_maps[variant], f"EFFSURF_{variant}", f"Effective-area weight, variant {variant}")
    plot_overlay_hist(
        weight_maps[variant],
        f"EFFSURF_{variant}",
        f"Effective-area weight, variant {variant}: area-weighted histograms (left), area with value >= x (right)",
    )

## 9. Differences between consecutive dust thresholds

As in notebooks 02, 03 and 05, a masked pixel counts as 0, so the difference maps contain the pixels gained or lost with the footprint. The table splits the change of $A_{\rm eff}$ (deg$^2$) into the change on the common pixels (better or worse depth and seeing), the pixels gained and the pixels lost. Variant `generic`.

In [ ]:
def diff_map(map_b, map_a):
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.where(mask_b, 0.0, np.ma.getdata(map_b).astype(float))
    a = np.where(mask_a, 0.0, np.ma.getdata(map_a).astype(float))
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_table(maps):
    rows = []
    for run_b, run_a in PAIRS:
        mb_, ma_ = maps[run_b], maps[run_a]
        vb, va = ~np.ma.getmaskarray(mb_), ~np.ma.getmaskarray(ma_)
        b = np.ma.getdata(mb_).astype(float)
        a = np.ma.getdata(ma_).astype(float)
        common, gained, lost = vb & va, vb & ~va, va & ~vb
        rows.append(
            {
                "pair (E(B-V) thresholds)": f"{EXTINCTION_CUT[run_b]:.3f} - {EXTINCTION_CUT[run_a]:.3f}",
                "area gained [deg2]": gained.sum() * pix_area,
                "area lost [deg2]": lost.sum() * pix_area,
                "dA_eff total [deg2]": b[vb].sum() - a[va].sum(),
                "from common pixels": (b[common] - a[common]).sum(),
                "from gained pixels": b[gained].sum(),
                "from lost pixels": -a[lost].sum(),
            }
        )
    return pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")


def plot_diff_mosaic(maps, tag, title):
    diffs = {pair: diff_map(maps[pair[0]], maps[pair[1]]) for pair in PAIRS}
    lims = [np.percentile(np.abs(d.compressed()), 99) for d in diffs.values() if d.count()]
    vlim = max(max(lims), 1e-6) if lims else 1.0
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diffs.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit="Δw",
            cbar=True,
        )
    fig.suptitle(
        f"{title}: differences between consecutive dust thresholds (common scale ±{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


plot_diff_mosaic(weight_maps["generic"], "EFFSURF_generic", "Effective-area weight (generic)")
pairs_generic = pair_table(eff_maps["generic"])
pairs_generic.to_csv(join(data_dir, "EffectiveSurfaceArea_generic_diff_summary.csv"))
display(pairs_generic.round(3))

## 10. Effective surface area versus the dust threshold

Left: $A_{\rm eff}$ and the geometric area. Middle: both normalized to the baseline run. Right: $A_{\rm eff}/A_{\rm geo}=\langle n_{\rm eff}\rangle/n_{\rm ref}$, the mean quality of the footprint, which separates the footprint size from the per-pixel effect.

In [ ]:
x = np.array([EXTINCTION_CUT[r] for r in RUN_NAMES])
i_ref = RUN_NAMES.index(REF_RUN)
styles = {"generic": ("tab:blue", "o"), "R2cut": ("tab:red", "s")}

fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
for variant, (color, marker) in styles.items():
    s = summaries[variant]
    axs[0].plot(x, s["A_eff [deg2]"].values, marker + "-", color=color, label=f"A_eff, {variant}")
    axs[1].plot(x, s["A_eff / baseline"].values, marker + "-", color=color, label=f"A_eff, {variant}")
    axs[2].plot(x, s["A_eff / A_geo"].values, marker + "-", color=color, label=variant)
s = summaries["generic"]
axs[0].plot(x, s["area_deg2"].values, "k^--", mfc="none", label="geometric area A_geo")
axs[1].plot(x, s["A_geo / baseline"].values, "k^--", mfc="none", label="geometric area A_geo")
axs[1].axhline(1.0, color="gray", ls="--", linewidth=0.8)
axs[0].set_ylabel("area [deg²]")
axs[1].set_ylabel("value / value of the baseline run")
axs[2].set_ylabel("A_eff / A_geo  =  <n_eff> / n_ref")
for ax, ref_y in zip(axs, (s["A_eff [deg2]"].values[i_ref], 1.0, s["A_eff / A_geo"].values[i_ref])):
    ax.plot(x[i_ref], ref_y, "o", mfc="none", mec="k", ms=12, label="baseline")
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axs[0].set_title(f"Effective surface area (n_ref = {N_REF:g} arcmin$^{{-2}}$)")
axs[1].set_title("Normalized to the baseline")
axs[2].set_title("Mean quality of the footprint")
fig.tight_layout()
save_fig(fig, "EFFSURF_vs_dust_threshold")
plt.show()

## Using the metric elsewhere

```python
metric = EffectiveSurfaceAreaMetric(nside=64, n_ref=27.0, extinction_cut=0.2, r_min=None)
bundle = mb.MetricBundle(metric, slicers.HealpixSlicer(nside=64, use_cache=False), sql_bands(),
                         maps_list=[maf_maps.DustMap(nside=64, interp=False)], run_name=run_name)
# after MetricBundleGroup.run_all():  A_eff = bundle_to_masked(bundle).sum()   [deg2]
```
`nside` of the metric must be the one of the slicer and of the dust map. `r_min = R2_MIN / (1 - R2_MIN)` gives the `R2cut` variant.

## Caveats

- $A_{\rm eff}$ inherits all the assumptions of the `n_eff` model of notebook 04 (galaxy population, colours, algorithm-dependent noise law) and the absolute value depends linearly on `N_REF`. Compare runs through the ratio to the baseline.
- The weight $n_{\rm eff}/n_{\rm ref}$ is not capped at 1: a pixel denser than the reference counts for more than its geometric area, which is what makes $A_{\rm eff}\propto N_{\rm eff,tot}$. Its ratio to the baseline therefore equals the `N_eff / baseline` of notebook 05; the value added is the area-equivalent interpretation and the MAF form.
- $A_{\rm eff}$ measures the shape-noise term only. The PSF-systematics term, the tomographic redshift distribution and the cosmic-variance part of the shear power spectrum are not included (see Section 12 of notebook 05).
- The seeing of a pixel is the root mean square of the per-visit `FWHM` (unweighted), and only the bands `r` and `i` enter the model, as in notebook 05.

## References
- Chang, C. et al. 2013, MNRAS 434, 2121, arXiv:1305.0793 (erratum: MNRAS 447, 1746, 2015)
- `04_NeffSeeingModel.ipynb` (model and calibration), `05_NeffMaps.ipynb` (n_eff maps), `03_WeakLensing.ipynb` (footprint), `01_compareExgalM5withCuts.ipynb` (depth)